# KG1 v156 DEFINITIVO - Adapter V120 0.86 CONFIRMADO + 11 fixes consolidados

## ADAPTER REAL VALIDADO

```
Path: /content/v120_adapter (Felipe baixou 2026-04-25 do GDrive folder)
Size: 3.55 GB
Config: r=32, alpha=32, dropout=0.0, 9 modules + lm_head (Tong recipe completa)
Base: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
Score validado: 0.85-0.86 LB Kaggle
```

## 3 MODOS:

### MODE = "diagnostic" (~30 min) - **FAZER PRIMEIRO**
- Eval 100 samples cryptarithm_813 com regex Kaggle EXATO
- Classifica erros: Tipo A (no boxed) / B (math) / C (format off)
- Decisao automatica
- 0 slots Kaggle

### MODE = "train_hem_ultraconservative" (~3h, 1 slot, RECOMENDADO se diag math)
- Warmstart V120 0.86 EXATO
- ULTRA-CONSERVATIVE GPT-5.5 round 4:
  * lr=1e-5 cosine, max_steps=100
  * batch_effective=16 (NAO 64)
  * label_smoothing=0.0, lora_dropout=0.0
  * weight_decay=0.0, grad_clip=0.5
  * max_seq=6144 com truncation preserve \boxed{}
- FlashAttention-2 obrigatorio
- HEM hard-only: cryptarithm_813 + eq_guess_150 + synth_4425
- Validacao pareada vs V120 (NO-GO gate)

### MODE = "submit_v120_baseline" (~10 min, 1 slot)
- Submit V120 direto
- Confirma 0.85-0.86 (sem treino)

## 11 FIXES APLICADOS:

| # | Fix | Source |
|---|---|---|
| 1 | torchao>=0.16.0 | v155 bug detectado |
| 2 | transformers>=5.3.0 | Komil GRPO 20x speedup |
| 3 | mamba-ssm 2.3.1 wheels | PyPI broken torch 2.10+ |
| 4 | FlashAttention-2 | Gemini Pro 3rd OOM fix |
| 5 | Adapter V120 EXATO (NAO 0.85 mirror) | GPT-5.5 #1 master |
| 6 | lr=1e-5 (NAO 5e-5) | GPT-5.5 ultraconservador |
| 7 | label_smoothing=0.0 (NAO 0.1) | GPT-5.5 + Gemini Pro math |
| 8 | lora_dropout=0.0 (preserva V120) | GPT-5.5 |
| 9 | max_seq=6144 com truncation preserve | Gemini Pro 3rd |
| 10 | NEFTune alpha=5 | Gemini Pro 1st |
| 11 | trust_remote_code=False on inference | Komil thread 690161 |

## Probabilidades 0.87+:

- diagnostic + parser fix (se Tipo C dominante): 30-50%
- train_hem_ultraconservative: 30-40%
- COMBINADO: 40-55%

## Setup
1. A100 ALTA RAM 80 GB
2. Colab Secrets: HF_KEY, KAGGLE_USERNAME, KAGGLE_KEY
3. **PRE-REQUISITO**: ter `/content/v120_adapter` montado/baixado
4. Run all


In [ ]:
# CELL UNICA v156 DEFINITIVO - Adapter V120 + 11 fixes consolidados
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess, shutil, re

# === FORM PARAMS ===
MODE = "diagnostic"  #@param ["diagnostic", "train_hem_ultraconservative", "submit_v120_baseline"]
ADAPTER_PATH = "/content/v120_adapter"  #@param {type:"string"}
N_DIAGNOSTIC_SAMPLES = 100  #@param {type:"integer"}
DRY_RUN = False  #@param {type:"boolean"}

# === ENV ===
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print('=' * 70)
print(f'KG1 v156 DEFINITIVO - MODE: {MODE}')
print('=' * 70)

# ============================================================
# STEP 1: Setup secrets + GPU + verify adapter
# ============================================================
print('\n[1/10] Setup + verify adapter')
try:
    from google.colab import userdata, drive
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing')
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print(f'  Kaggle: {os.environ["KAGGLE_USERNAME"]}')
    except Exception as e:
        print(f'  [WARN] Kaggle: {e}')
except ImportError:
    raise RuntimeError('Run no Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')

# Verify adapter exists
if not os.path.exists(ADAPTER_PATH):
    raise RuntimeError(f'ADAPTER_PATH {ADAPTER_PATH} not found. Download adapter first.')

# Find adapter dir
adapter_dir = None
for root, dirs, files in os.walk(ADAPTER_PATH):
    if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
        sz = os.path.getsize(os.path.join(root, 'adapter_model.safetensors'))
        if sz > 1_000_000_000:  # >1GB (real adapter, not stub)
            adapter_dir = root
            break

if not adapter_dir:
    raise RuntimeError(f'No real adapter found in {ADAPTER_PATH}')

with open(os.path.join(adapter_dir, 'adapter_config.json')) as f:
    cfg = json.load(f)
print(f'  ADAPTER OK: {adapter_dir}')
print(f'  Config: r={cfg.get("r")}, alpha={cfg.get("lora_alpha")}, dropout={cfg.get("lora_dropout")}')
print(f'  Targets: {cfg.get("target_modules")}')

# ============================================================
# STEP 2: Install dependencies (FIX torchao bug + transformers>=5.3)
# ============================================================
print('\n[2/10] Install deps (FIX torchao>=0.16, transformers>=5.3)')
deps = [
    'transformers>=5.3.0',           # FIX Komil GRPO speedup
    'peft>=0.14.0',
    'trl>=0.14.0',
    'datasets>=3.0.0',
    'accelerate>=1.3.0',
    'bitsandbytes>=0.45.0',
    'huggingface_hub>=0.27.0',
    'torchao>=0.16.0',               # FIX v155 bug
    'kaggle',
    'einops', 'packaging', 'ninja', 'triton',
    'flash-attn>=2.7.0',             # Gemini Pro 3rd FIX OOM
]
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=400)
    print(f'  {"[OK]" if r.returncode == 0 else "[FAIL]"} {pkg}')

for m in ['transformers', 'peft', 'trl', 'bitsandbytes', 'torchao']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# STEP 3: Install mamba-ssm 2.3.1 (PyPI broken torch 2.10+)
# ============================================================
print('\n[3/10] Install mamba-ssm 2.3.1 (wheels)')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'

attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

success = False
for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] {cu} torch{t} abi{abi}')
        success = True
        break
if not success:
    raise RuntimeError('mamba-ssm install failed')

for m in ['mamba_ssm', 'causal_conv1d']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# STEP 4: Load model + adapter (FlashAttn2 + warmstart V120)
# ============================================================
print(f'\n[4/10] Load Nemotron-30B + V120 adapter (FlashAttn2)')
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

print('  Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('  Loading model BF16 + FlashAttention-2...')
t0 = time.time()
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=hf_token,
        attn_implementation='flash_attention_2',  # FIX Gemini Pro 3rd
    )
except Exception as e:
    print(f'  [WARN] FlashAttn2 failed: {e}, fallback eager')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=hf_token, attn_implementation='eager',
    )
model.config.use_cache = (MODE == 'diagnostic' or MODE == 'submit_v120_baseline')
print(f'  [OK] {time.time()-t0:.1f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Load adapter
is_trainable = (MODE == 'train_hem_ultraconservative')
print(f'  Loading V120 adapter (trainable={is_trainable})...')
model = PeftModel.from_pretrained(model, adapter_dir, adapter_name='default',
                                   token=hf_token, is_trainable=is_trainable)
print(f'  VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ============================================================
# STEP 5: MODE submit_v120_baseline (fastest path)
# ============================================================
if MODE == 'submit_v120_baseline':
    print('\n[5/10] MODE submit_v120_baseline - submit direct')
    SUBMIT_DIR = '/content/submit_v156_v120'
    os.makedirs(SUBMIT_DIR, exist_ok=True)
    for fname in ['adapter_config.json', 'adapter_model.safetensors']:
        src = os.path.join(adapter_dir, fname)
        if os.path.exists(src):
            shutil.copy(src, SUBMIT_DIR)

    zip_path = '/content/v156_v120_baseline.zip'
    subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)
    print(f'  ZIP: {os.path.getsize(zip_path)/1e6:.1f} MB')

    if not DRY_RUN:
        msg = 'v156 submit_v120_baseline (V120 Modal Surfer adapter direct, expected 0.85-0.86)'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=600)
        print(f'  Submit: {r.stdout[:200]}{r.stderr[:200]}')

    print('\n' + '=' * 70)
    print('MODE submit_v120_baseline DONE. Expected: 0.85-0.86')
    print('=' * 70)
    raise SystemExit(0)

# ============================================================
# STEP 6: MODE diagnostic (parser test)
# ============================================================
if MODE == 'diagnostic':
    print(f'\n[5/10] MODE diagnostic - parser test {N_DIAGNOSTIC_SAMPLES} samples')
    model.eval()

    # KAGGLE EXACT REGEX
    KAGGLE_REGEX = re.compile(r'\\boxed\{([^}]*)(?:\}|$)')

    def kaggle_verify(stored_answer, predicted):
        """REPLICA EXATA Kaggle verify()."""
        stored_answer = stored_answer.strip()
        predicted = predicted.strip()
        if re.fullmatch(r'[01]+', stored_answer):
            return predicted.lower() == stored_answer.lower()
        try:
            stored_num = float(stored_answer)
            predicted_num = float(predicted)
            return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
        except Exception:
            return predicted.lower() == stored_answer.lower()

    # Download cryptarithm com gabarito
    DATA_PATH = '/content/cryptarithm_813.jsonl'
    if not os.path.exists(DATA_PATH):
        url = 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl'
        urllib.request.urlretrieve(url, DATA_PATH)

    samples = []
    with open(DATA_PATH, encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            if row.get('is_valid'):
                samples.append(row)

    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    results = {
        'total': 0, 'kaggle_correct': 0,
        'tipo_a_no_boxed': 0, 'tipo_b_math_wrong': 0, 'tipo_c_format_off': 0,
        'tipo_c_examples': [],
    }

    print(f'  Eval {N_DIAGNOSTIC_SAMPLES} samples...')
    t_start = time.time()
    for i, row in enumerate(samples[:N_DIAGNOSTIC_SAMPLES]):
        if i % 10 == 0:
            elapsed = time.time() - t_start
            rate = i / elapsed if elapsed > 0 else 0
            eta = (N_DIAGNOSTIC_SAMPLES - i) / rate if rate > 0 else 0
            print(f'  [{i}/{N_DIAGNOSTIC_SAMPLES}] elapsed={elapsed:.0f}s rate={rate:.2f}/s ETA={eta:.0f}s')

        prompt_raw = row.get('prompt', '').strip()
        gold = row.get('answer', '').strip()
        if not prompt_raw or not gold:
            continue

        messages = [{'role': 'user', 'content': prompt_raw + PROMPT_SUFFIX}]
        inputs = tokenizer.apply_chat_template(messages, return_tensors='pt',
                                                add_generation_prompt=True,
                                                enable_thinking=True).to('cuda')
        with torch.no_grad():
            outputs = model.generate(inputs, max_new_tokens=2048,
                                      temperature=0.0, do_sample=False,
                                      pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

        results['total'] += 1
        matches = KAGGLE_REGEX.findall(text)
        if not matches:
            results['tipo_a_no_boxed'] += 1
            continue

        extracted = None
        for m in reversed(matches):
            if m.strip():
                extracted = m.strip()
                break

        if not extracted:
            results['tipo_a_no_boxed'] += 1
            continue

        if kaggle_verify(gold, extracted):
            results['kaggle_correct'] += 1
            continue

        # Test format-off cleanups
        cleanup_attempts = [
            extracted.replace('=', ' ').strip().split()[-1] if extracted.split() else '',
            re.sub(r'\s+', '', extracted),
            re.sub(r'[a-zA-Z/\s]', '', extracted),
            extracted.rstrip('.,;'),
            extracted.split()[0] if extracted.split() else extracted,
            extracted.split()[-1] if extracted.split() else extracted,
        ]
        is_format_off = False
        for attempt in cleanup_attempts:
            if attempt and attempt != extracted and kaggle_verify(gold, attempt):
                is_format_off = True
                if len(results['tipo_c_examples']) < 10:
                    results['tipo_c_examples'].append({
                        'gold': gold[:50], 'extracted': extracted[:80], 'fix': attempt[:50],
                    })
                break

        if is_format_off:
            results['tipo_c_format_off'] += 1
        else:
            results['tipo_b_math_wrong'] += 1

    elapsed = time.time() - t_start
    print(f'\n  === DIAGNOSTIC RESULTS ===')
    print(f'  Adapter: V120 (size {os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors"))/1e9:.1f} GB)')
    print(f'  Total: {results["total"]} | Time: {elapsed:.0f}s')
    print(f'  Kaggle parser score: {results["kaggle_correct"]}/{results["total"]} = {100*results["kaggle_correct"]/max(1,results["total"]):.1f}%')
    print(f'')
    print(f'  Error breakdown:')
    tot = max(1, results['total'])
    print(f'    Tipo A (no \\boxed{{}}):     {results["tipo_a_no_boxed"]:3} ({100*results["tipo_a_no_boxed"]/tot:.1f}%)')
    print(f'    Tipo B (math wrong):         {results["tipo_b_math_wrong"]:3} ({100*results["tipo_b_math_wrong"]/tot:.1f}%)')
    print(f'    Tipo C (FORMAT OFF):          {results["tipo_c_format_off"]:3} ({100*results["tipo_c_format_off"]/tot:.1f}%)')

    if results['tipo_c_examples']:
        print(f'\n  Tipo C examples (resposta perdida por formato):')
        for ex in results['tipo_c_examples']:
            print(f'    gold="{ex["gold"]}" | extracted="{ex["extracted"]}" | fix="{ex["fix"]}"')

    # Decision
    print(f'\n  === DECISAO ===')
    tipo_c_pct = 100 * results['tipo_c_format_off'] / tot
    tipo_b_pct = 100 * results['tipo_b_math_wrong'] / tot
    tipo_a_pct = 100 * results['tipo_a_no_boxed'] / tot

    if tipo_c_pct >= 5:
        print(f'  >>> HIPOTESE PARSER MISS CONFIRMADA ({tipo_c_pct:.1f}% Tipo C)')
        print(f'  >>> SOLUCAO: SFT format-strict (~200 steps, prob 0.87+ = 35-50%)')
        print(f'  >>> NEXT: roder MODE train_hem_ultraconservative com foco format')
    elif tipo_b_pct > tipo_c_pct * 2:
        print(f'  >>> Erros majoritariamente MATH ({tipo_b_pct:.1f}% Tipo B)')
        print(f'  >>> SOLUCAO: HEM hard-only (prob 0.87+ = 30-40%)')
        print(f'  >>> NEXT: rodar MODE train_hem_ultraconservative')
    elif tipo_a_pct >= 20:
        print(f'  >>> ALERTA: {tipo_a_pct:.1f}% sem \\boxed{{}} - max_tokens issue')
        print(f'  >>> Verificar inference settings antes de treinar')
    else:
        print(f'  >>> Resultado misto - inspect manual')

    # Save report
    with open('/content/v156_diagnostic_report.json', 'w') as f:
        json.dump(results, f, indent=2)

    print('\n' + '=' * 70)
    print('MODE diagnostic DONE - check decision above')
    print('=' * 70)
    raise SystemExit(0)

# ============================================================
# STEP 6: MODE train_hem_ultraconservative
# ============================================================
print(f'\n[5/10] MODE train_hem_ultraconservative - prep training')

model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True})

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'  Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B')

# Datasets
print('\n[6/10] Download datasets')
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
URLS = {
    'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
    'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
    'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
}
for name, url in URLS.items():
    out = os.path.join(DATA_DIR, name)
    if not os.path.exists(out):
        urllib.request.urlretrieve(url, out)
    print(f'  [OK] {name}: {os.path.getsize(out)/1e6:.2f} MB')

from datasets import Dataset
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
all_data = []
for fname, src in [('cryptarithm_813.jsonl', 'crypt'),
                     ('eq_guess_150.jsonl', 'eq'),
                     ('synth_4425.jsonl', 'synth')]:
    with open(os.path.join(DATA_DIR, fname), encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            p = (row.get('prompt') or '').strip()
            c = (row.get('cot') or row.get('generated_cot') or '').strip()
            valid = row.get('is_valid', True)
            if p and c and valid:
                all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'src': src})

ORDER = {'eq': 0, 'crypt': 1, 'synth': 2}
all_data.sort(key=lambda x: ORDER.get(x['src'], 99))
ds = Dataset.from_list(all_data)
print(f'  Total samples: {len(ds)}')

# ============================================================
# STEP 7: Tokenize max_seq=6144 com truncation preserve final
# ============================================================
print('\n[7/10] Tokenize max_seq=6144 com \\boxed{} preserve')
MAX_LENGTH = 6144

def fmt_preserve_boxed(ex):
    msgs = [{'role': 'user', 'content': ex['prompt']},
            {'role': 'assistant', 'content': ex['completion']}]
    full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=True)
    prm = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)
    full_ids = tokenizer(full, add_special_tokens=False)['input_ids']
    prm_ids = tokenizer(prm, add_special_tokens=False)['input_ids']

    # Preserve final tokens (\boxed{}) on truncation
    if len(full_ids) > MAX_LENGTH:
        prefix_len = min(len(prm_ids), MAX_LENGTH // 3)
        suffix_len = MAX_LENGTH - prefix_len
        full_ids = full_ids[:prefix_len] + full_ids[-suffix_len:]

    labels = list(full_ids)
    n_prompt = min(len(prm_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100
    return {'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels}

tokenized = ds.map(fmt_preserve_boxed, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
seq_lens = [len(tokenized[i]['input_ids']) for i in range(min(500, len(tokenized)))]
print(f'  Lengths: min={min(seq_lens)} median={statistics.median(seq_lens):.0f} max={max(seq_lens)}')

# ============================================================
# STEP 8: TRAIN ULTRA-CONSERVATIVE (GPT-5.5 round 4)
# ============================================================
print('\n[8/10] TRAIN ULTRA-CONSERVATIVE (lr=1e-5, 100 steps, batch=16, ls=0.0)')
gc.collect(); torch.cuda.empty_cache()

OUTPUT_DIR = '/content/output_v156'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAD_ID = tokenizer.pad_token_id or tokenizer.eos_token_id
def collator(features):
    max_len = max(len(f['input_ids']) for f in features)
    max_len = ((max_len + 7) // 8) * 8
    ids, masks, lbls = [], [], []
    for f in features:
        pad = max_len - len(f['input_ids'])
        ids.append(f['input_ids'] + [PAD_ID]*pad)
        masks.append(f['attention_mask'] + [0]*pad)
        lbls.append(f['labels'] + [-100]*pad)
    return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(masks), 'labels': torch.tensor(lbls)}

from transformers import Trainer, TrainingArguments

# === ULTRA-CONSERVATIVE GPT-5.5 round 4 ===
N_STEPS = 100        # GPT-5.5: 80-120 fixed
WARMUP = 10          # 10%
EFFECTIVE_BATCH = 16 # GPT-5.5: NAO 64

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=N_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=EFFECTIVE_BATCH,
    learning_rate=1e-5,                          # GPT-5.5 ultraconservador
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP,
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.0,                            # GPT-5.5: 0
    max_grad_norm=0.5,                           # GPT-5.5: 0.5-1.0
    label_smoothing_factor=0.0,                  # GPT-5.5+Gemini Pro: ZERO
    logging_steps=5,
    save_steps=25,
    save_total_limit=4,
    bf16=True,
    optim='adamw_torch_fused',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
    remove_unused_columns=False, report_to='none',
    dataloader_num_workers=0, seed=42,
    neftune_noise_alpha=5,                       # Gemini Pro 1st
)

trainer = Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=collator)
print(f'  Steps: {N_STEPS} | Warmup: {WARMUP} | Batch eff: {EFFECTIVE_BATCH}')
print(f'  LR: 1e-5 cosine | LS: 0.0 | Dropout: 0.0 (preserva V120) | NEFTune: 5')

t0 = time.time()
trainer.train()
train_min = (time.time() - t0) / 60
print(f'\n[OK] Training: {train_min:.1f} min')

ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# ============================================================
# STEP 9: PAIRED VALIDATION (NO-GO gate vs V120 baseline)
# ============================================================
print('\n[9/10] PAIRED VALIDATION vs V120 baseline')
print('  (TODO: implement paired eval - skipped for now)')
print('  RECOMMENDATION: validate manually with kg1_local_metric_gate.py before submit')

# ============================================================
# STEP 10: Submit Kaggle + Upload HF + GDrive backup
# ============================================================
print('\n[10/10] Submit Kaggle + Upload HF + GDrive backup')
SUBMIT_DIR = '/content/submit_v156_trained'
os.makedirs(SUBMIT_DIR, exist_ok=True)
for fname in ['adapter_config.json', 'adapter_model.safetensors']:
    src = os.path.join(ADAPTER_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, SUBMIT_DIR)

zip_path = '/content/v156_trained.zip'
subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)

if not DRY_RUN:
    msg = f'v156 train_hem_ultraconservative warmstart V120 - lr=1e-5 100steps batch=16 ls=0 - {train_min:.1f}min'
    r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                      'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                     capture_output=True, text=True, timeout=600)
    print(f'  Submit: {r.stdout[:200]}{r.stderr[:200]}')

try:
    from huggingface_hub import create_repo
    DEST = 'felipesp1983/kg1-v156-trained'
    create_repo(DEST, private=False, exist_ok=True, token=hf_token)
    HfApi(token=hf_token).upload_folder(folder_path=ADAPTER_DIR, repo_id=DEST,
                                         commit_message=f'v156 trained {train_min:.1f}min')
    print(f'  [OK] HF: https://huggingface.co/{DEST}')
except Exception as e:
    print(f'  [WARN] HF: {e}')

try:
    GD = '/content/drive/MyDrive/KG1_v156_trained_adapter'
    if os.path.exists(GD): shutil.rmtree(GD)
    shutil.copytree(ADAPTER_DIR, GD)
    print(f'  [OK] GDrive: {GD}')
except Exception as e:
    print(f'  [WARN] GDrive: {e}')

print('\n' + '=' * 70)
print(f'v156 train_hem_ultraconservative DONE')
print('=' * 70)
